Quick EDA of Olist_customer fields


In [27]:
import sys
import subprocess
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

# Load the Texas ZIP centroids dataset
data_path = Path("Texas_ZIP_Geo") / "texas_zip_centroids.csv"
texas_df = pd.read_csv(data_path)

print(f"Loaded {len(texas_df):,} rows and {texas_df.shape[1]} columns from: {data_path}")
display(texas_df.head())

# Unique counts for requested fields
unique_zip = texas_df["zip"].nunique(dropna=True)
unique_county = texas_df["county_name"].nunique(dropna=True)

print(f"Unique ZIP values: {unique_zip:,}")
print(f"Unique county_name values: {unique_county:,}")

# Optional context: top values
print("\nTop 10 county names by frequency:")
display(texas_df["county_name"].value_counts(dropna=False).head(10))

Loaded 1,985 rows and 10 columns from: Texas_ZIP_Geo\texas_zip_centroids.csv


,zip,latitude,longitude,city,state,county_geoid,county_name,population,density,centroid_source
0,73960,36.49167,-101.79265,TEXHOMA,TX,48421,Sherman,5,3.2,uszips
1,75001,32.96015,-96.83808,ADDISON,TX,48113,Dallas,16872,1736.0,uszips
2,75002,33.08946,-96.60639,ALLEN,TX,48085,Collin,75057,771.1,uszips
3,75006,32.96165,-96.89717,CARROLLTON,TX,48113,Dallas,47807,1081.2,uszips
4,75007,33.00498,-96.89590,CARROLLTON,TX,48121,Denton,54646,1772.1,uszips


Unique ZIP values: 1,985
Unique county_name values: 254

Top 10 county names by frequency:


county_name
Harris     132
Dallas      84
Bexar       70
Tarrant     67
Travis      47
El Paso     30
Collin      29
Hidalgo     29
Denton      27
Nueces      25
Name: count, dtype: int64

Checking for missing or incorrect values

In [28]:
missing_population = texas_df[texas_df["population"].isna()]
missing_density = texas_df[texas_df["density"].isna()]

print("ZIPs with missing population:", len(missing_population))
print("ZIPs with missing density:", len(missing_density))

display(missing_population[["zip", "population", "density"]])
display(missing_density[["zip", "population", "density"]])

ZIPs with missing population: 0
ZIPs with missing density: 0


,zip,population,density


,zip,population,density


In [29]:
#Removing ZIP codes with zero population or density
texas_df = texas_df[(texas_df["population"] > 0) & (texas_df["density"] > 0)]

zero_population = texas_df[texas_df["population"] == 0]
zero_density = texas_df[texas_df["density"] == 0]

print("ZIPs with zero population:", len(zero_population))
print("ZIPs with zero density:", len(zero_density))

display(zero_population[["zip", "population", "density"]])
display(zero_density[["zip", "population", "density"]])

ZIPs with zero population: 0
ZIPs with zero density: 0


,zip,population,density


,zip,population,density


In [30]:
negative_population = texas_df[texas_df["population"] < 0]
negative_density = texas_df[texas_df["density"] < 0]

print("ZIPs with negative population:", len(negative_population))
print("ZIPs with negative density:", len(negative_density))

display(negative_population[["zip", "population", "density"]])
display(negative_density[["zip", "population", "density"]])

ZIPs with negative population: 0
ZIPs with negative density: 0


,zip,population,density


,zip,population,density


In [31]:
median_density = texas_df["density"].median()    
print(f"\nMedian population density across all ZIP codes: {median_density:.2f} people/sq mile")


Median population density across all ZIP codes: 32.40 people/sq mile


In [32]:
# Establish ZIP target-weight formula using population and density per ZIP code
population_series = pd.to_numeric(texas_df["population"], errors="coerce").fillna(0)
density_series = pd.to_numeric(texas_df["density"], errors="coerce").fillna(0)

density_modifier = density_series.div(median_density).clip(lower=0).pow(0.25)
texas_df["target_weight"] = population_series * density_modifier

total_target_weight = texas_df["target_weight"].sum()
texas_df["target_zip_proportion"] = texas_df["target_weight"] / total_target_weight

print(f"Total target weight across all Texas ZIP codes: {total_target_weight:,.2f}")
display(
    texas_df[["zip", "population", "density", "target_weight", "target_zip_proportion"]]
    .sort_values("target_weight", ascending=False)
    .head(10)
)


Total target weight across all Texas ZIP codes: 61,624,228.48


,zip,population,density,target_weight,target_zip_proportion
1041,77494,140157,1428.4,361152.812135,0.005861
1004,77449,130028,1856.7,357755.290123,0.005805
1538,78660,124834,1080.5,299987.270778,0.004868
900,77084,110217,1341.0,279556.456965,0.004536
1983,79936,102991,1535.4,270220.570472,0.004385
990,77433,116550,763.1,256756.465572,0.004166
31,75052,97278,1420.3,250307.234683,0.004062
1085,77573,98109,1271.8,245571.126068,0.003985
974,77407,88319,1705.7,237899.512937,0.003860
853,77036,71770,3809.2,236327.977126,0.003835


In [33]:
# Quick validation of the weight distribution
print(f"Target weight proportions sum to: {texas_df['target_zip_proportion'].sum():.12f}")
display(texas_df[["target_weight", "target_zip_proportion"]].describe())


Target weight proportions sum to: 1.000000000000


,target_weight,target_zip_proportion
count,1915.000000,1.915000e+03
mean,32179.753774,5.221932e-04
std,49361.853749,8.010137e-04
min,1.681793,2.729110e-08
25%,720.862704,1.169772e-05
50%,6362.298465,1.032435e-04
75%,47211.651591,7.661216e-04
max,361152.812135,5.860565e-03


Grouping Olist customer by State-ZIP

In [34]:
olist_customers_path = Path("Olist E-Commerce Dataset") / "olist_customers_with_source_code.csv"
olist_customers_df = pd.read_csv(olist_customers_path)
total_olist_customers = len(olist_customers_df)

print(f"Loaded {total_olist_customers:,} Olist customers from: {olist_customers_path}")


Loaded 99,441 Olist customers from: Olist E-Commerce Dataset\olist_customers_with_source_code.csv


In [35]:
texas_df["target_zip_capacity"] = texas_df["target_zip_proportion"] * total_olist_customers

display(
    texas_df[["zip", "target_weight", "target_zip_proportion", "target_zip_capacity"]]
    .sort_values("target_zip_capacity", ascending=False)
    .head(10)
)


,zip,target_weight,target_zip_proportion,target_zip_capacity
1041,77494,361152.812135,0.005861,582.780469
1004,77449,357755.290123,0.005805,577.297999
1538,78660,299987.270778,0.004868,484.079638
900,77084,279556.456965,0.004536,451.111102
1983,79936,270220.570472,0.004385,436.046088
990,77433,256756.465572,0.004166,414.319503
31,75052,250307.234683,0.004062,403.912590
1085,77573,245571.126068,0.003985,396.270086
974,77407,237899.512937,0.003860,383.890656
853,77036,236327.977126,0.003835,381.354720


In [36]:
# Final checks on the proportional allocation
capacity_total = texas_df["target_zip_capacity"].sum()

print(f"Target ZIP proportion total: {texas_df['target_zip_proportion'].sum():.12f}")
print(f"Target ZIP capacity total: {capacity_total:,.6f}")
print(f"Total Olist customers: {total_olist_customers:,}")

display(
    texas_df[["zip", "population", "density", "target_weight", "target_zip_proportion", "target_zip_capacity"]]
    .head(10)
)


Target ZIP proportion total: 1.000000000000
Target ZIP capacity total: 99,441.000000
Total Olist customers: 99,441


,zip,population,density,target_weight,target_zip_proportion,target_zip_capacity
0,73960,5,3.2,2.802988,4.548516e-08,0.004523
1,75001,16872,1736.0,45647.578525,7.407408e-04,73.660003
2,75002,75057,771.1,165780.186955,2.690179e-03,267.514060
3,75006,47807,1081.2,114903.100657,1.864577e-03,185.415372
4,75007,54646,1772.1,148608.681712,2.411530e-03,239.804964
5,75009,34260,134.0,48857.057317,7.928222e-04,78.839034
6,75010,33326,1888.8,92085.962322,1.494314e-03,148.596103
7,75013,49796,1278.1,124795.640521,2.025107e-03,201.378639
8,75019,44917,1009.1,106110.421844,1.721895e-03,171.226914
9,75020,24835,157.1,36852.924552,5.980265e-04,59.468358


# Validation Checks

In [37]:
# Safe numeric versions
texas_df["population"] = pd.to_numeric(texas_df["population"], errors="coerce")
texas_df["density"] = pd.to_numeric(texas_df["density"], errors="coerce")

validation_summary = {
    "total_zips": len(texas_df),
    "missing_population": texas_df["population"].isna().sum(),
    "missing_density": texas_df["density"].isna().sum(),
    "zero_population": (texas_df["population"] == 0).sum(),
    "zero_density": (texas_df["density"] == 0).sum(),
    "negative_population": (texas_df["population"] < 0).sum(),
    "negative_density": (texas_df["density"] < 0).sum(),
}

# Only check these if the columns already exist
if "target_weight" in texas_df.columns:
    validation_summary["missing_target_weight"] = texas_df["target_weight"].isna().sum()
    validation_summary["zero_target_weight"] = (texas_df["target_weight"] == 0).sum()

if "target_zip_capacity" in texas_df.columns:
    validation_summary["missing_target_zip_capacity"] = texas_df["target_zip_capacity"].isna().sum()
    validation_summary["zero_target_zip_capacity"] = (texas_df["target_zip_capacity"] == 0).sum()

pd.Series(validation_summary)

total_zips                     1915
missing_population                0
missing_density                   0
zero_population                   0
zero_density                      0
negative_population               0
negative_density                  0
missing_target_weight             0
zero_target_weight                0
missing_target_zip_capacity       0
zero_target_zip_capacity          0
dtype: int64

In [38]:
problem_zips = texas_df[
    texas_df["population"].isna()
    | texas_df["density"].isna()
    | (texas_df["population"] <= 0)
    | (texas_df["density"] <= 0)
]

display(problem_zips[["zip", "population", "density"]])

,zip,population,density


In [39]:
problem_weight_zips = texas_df[
    texas_df["target_weight"].isna()
    | (texas_df["target_weight"] <= 0)
    | texas_df["target_zip_capacity"].isna()
    | (texas_df["target_zip_capacity"] <= 0)
]

display(
    problem_weight_zips[
        ["zip", "population", "density", "target_weight", "target_zip_proportion", "target_zip_capacity"]
    ]
)

,zip,population,density,target_weight,target_zip_proportion,target_zip_capacity


In [40]:
sorted_texas_df = texas_df.sort_values("target_zip_capacity", ascending=False)
sorted_texas_df.head(10)

,zip,latitude,longitude,city,state,county_geoid,county_name,population,density,centroid_source,target_weight,target_zip_proportion,target_zip_capacity
1041,77494,29.74566,-95.82302,KATY,TX,48157,Fort Bend,140157,1428.4,uszips,361152.812135,0.005861,582.780469
1004,77449,29.83674,-95.73547,KATY,TX,48201,Harris,130028,1856.7,uszips,357755.290123,0.005805,577.297999
1538,78660,30.44361,-97.59558,PFLUGERVILLE,TX,48453,Travis,124834,1080.5,uszips,299987.270778,0.004868,484.079638
900,77084,29.82698,-95.66120,HOUSTON,TX,48201,Harris,110217,1341.0,uszips,279556.456965,0.004536,451.111102
1983,79936,31.77373,-106.29631,EL PASO,TX,48141,El Paso,102991,1535.4,uszips,270220.570472,0.004385,436.046088
990,77433,29.94920,-95.73979,CYPRESS,TX,48201,Harris,116550,763.1,uszips,256756.465572,0.004166,414.319503
31,75052,32.66455,-97.02809,GRAND PRAIRIE,TX,48113,Dallas,97278,1420.3,uszips,250307.234683,0.004062,403.912590
1085,77573,29.50213,-95.08636,LEAGUE CITY,TX,48167,Galveston,98109,1271.8,uszips,245571.126068,0.003985,396.270086
974,77407,29.67658,-95.71455,RICHMOND,TX,48157,Fort Bend,88319,1705.7,uszips,237899.512937,0.003860,383.890656
853,77036,29.70119,-95.53659,HOUSTON,TX,48201,Harris,71770,3809.2,uszips,236327.977126,0.003835,381.354720


In [41]:
sorted_texas_df.to_csv("texas_target_weights.csv", index=False)